# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa 
Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata (avoid subscripting, show as JSON for exploration)
# For demonstration, print title and description from metadata
meta_json = dataset.metadata.to_json()
print(f"Dataset: {meta_json['name']}")
print(meta_json['description'])

## 2. Data Overview

Review available record sets, fields, and their `@id` values. This step helps to understand the structure and what data is present for extraction.

In [ ]:
from pprint import pprint

# Get the list of record sets (by their @id)
record_sets = dataset.metadata.list_record_sets()
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")

# For each record set, print its fields (by @id and name)
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs['@id']}, Name: {rs.get('name','')}")
    fields = dataset.metadata.list_fields(record_set=rs['@id'])
    print("  Fields:")
    for f in fields:
        print(f"    - @id: {f['@id']}, name: {f.get('name','')}")

## 3. Data Extraction

Load data from each record set into a DataFrame using `mlcroissant`. Always reference record sets and fields/columns by their `@id`.

In [ ]:
# Define record set IDs you discovered
record_set_ids = [rs['@id'] for rs in dataset.metadata.list_record_sets()]

dataframes = {}

for rs_id in record_set_ids:
    print(f"Extracting records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

# For demonstration, pick the first record set for further analysis
main_record_set_id = record_set_ids[0]
# Display its columns
print(f"Columns for {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common EDA steps: filter, normalize, and group data. Reference all elements by their `@id`.

In [ ]:
import numpy as np

# Select a numeric field by its @id (pick a scoring field, e.g., GAD-7, PHQ-9, or PSQ if present)
# Inspect available numeric fields
fields = dataset.metadata.list_fields(record_set=main_record_set_id)
numeric_fields = [f for f in fields if f.get('dataType') in ['Float', 'Integer', 'Number']]

print("Numeric fields in this record set:")
for nf in numeric_fields:
    print(f"- {nf['@id']}: {nf.get('name', '')}")

# For demo, use the first numeric field
if len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]['@id']
else:
    raise ValueError('No numeric field found for EDA.')

# Filter out rows where that score is above a threshold
threshold = 10
df = dataframes[main_record_set_id]
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())
    
    # Normalize the numeric field (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    )/filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a grouping field (e.g. by demographic field)
    # Try to find a categorical field
    group_fields = [f for f in fields if f.get('dataType') == 'Text']
    if len(group_fields) > 0:
        group_field_id = group_fields[0]['@id']
        if group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped filtered data by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print('No suitable group_by field found.')
else:
    print(f"Field '{numeric_field_id}' not present in DataFrame columns.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized version.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id in df.columns:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')

    if f"{numeric_field_id}_normalized" in filtered_df:
        plt.subplot(1,2,2)
        filtered_df[f"{numeric_field_id}_normalized"].hist(bins=15)
        plt.title(f"Normalized {numeric_field_id} (> {threshold})")
        plt.xlabel(f"{numeric_field_id}_normalized")

    plt.tight_layout()
    plt.show()
else:
    print(f"No numeric data to visualize for field: {numeric_field_id}")

## 6. Conclusion

- The dataset provides multiple record sets, likely including core survey and demographics data.
- We demonstrated loading data, filtering on a numeric field (by @id), normalization, grouping, and visualization.
- Fields, columns, and record sets are referenced by their Croissant `@id`, ensuring robust and schema-compliant exploration.

You can adapt the steps to drill into specific mental health measures such as GAD-7, PHQ-9, or PSQ, or conduct more advanced analyses.

Refer to the dataset's Croissant schema for authoritative field semantics.